In [50]:
from pathlib import Path
input_dir_path = Path("/") / "kaggle" / "input" / "competitions" / "nlp-getting-started"
!ls {input_dir_path}

sample_submission.csv  test.csv  train.csv


In [51]:
import pandas as pd

In [52]:
train_df = pd.read_csv(input_dir_path / "train.csv")

In [53]:
train_df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [54]:
train_df.describe(include='object')

,keyword,location,text
count,7552,5080,7613
unique,221,3341,7503
top,fatalities,USA,11-Year-Old Boy Charged With Manslaughter of T...
freq,45,104,10


In [55]:
from datasets import Dataset, DatasetDict

In [56]:
dataset_ = Dataset.from_pandas(train_df)
dataset_

Dataset({
    features: ['id', 'keyword', 'location', 'text', 'target'],
    num_rows: 7613
})

In [57]:
# model_name = 'distilbert/distilbert-base-uncased-finetuned-sst-2-english'
model_name = 'microsoft/deberta-v3-small'

In [58]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [59]:
def tok_func(x): return tokenizer(x['text'])

In [60]:
dataset_tokenized = dataset_.map(tok_func, batched=True)

Map:   0%|          | 0/7613 [00:00<?, ? examples/s]

In [61]:
row = dataset_tokenized[0]
row['text'], row['input_ids']

('Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all',
 [581,
  65453,
  281,
  262,
  18037,
  265,
  291,
  953,
  117831,
  903,
  4924,
  17018,
  43632,
  381,
  305])

In [62]:
dataset_tokenized = dataset_tokenized.rename_columns({'target': 'labels', 'text': 'input'})

In [63]:
dictionary_dataset_tokenized = dataset_tokenized.train_test_split(0.25, seed=42)
dictionary_dataset_tokenized

DatasetDict({
    train: Dataset({
        features: ['id', 'keyword', 'location', 'input', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5709
    })
    test: Dataset({
        features: ['id', 'keyword', 'location', 'input', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1904
    })
})

In [64]:
eval_df = pd.read_csv(input_dir_path / "test.csv")
eval_dataset = Dataset.from_pandas(eval_df).map(tok_func, batched=True).rename_columns({'text': 'input'})

Map:   0%|          | 0/3263 [00:00<?, ? examples/s]

In [65]:
from transformers import TrainingArguments,Trainer
import numpy as np
from sklearn.metrics import f1_score

In [66]:
def metric(x): 
    predictions, labels = x
    preds = np.argmax(predictions, axis=1)
    return {'f1': f1_score(preds, labels)}

In [80]:
batch_size = 32
epochs = 4
learning_rate = 2e-7

In [81]:
args = TrainingArguments(
    'outputs',
    learning_rate=learning_rate,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    fp16=False,
    bf16=False,
    per_device_train_batch_size=batch_size, 
    per_device_eval_batch_size=batch_size*2,
    num_train_epochs=epochs,
    weight_decay=0.01,
    report_to='none',
    eval_strategy='epoch',
    logging_strategy='epoch',
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [82]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
trainer = Trainer(
    model, 
    args, 
    train_dataset=dictionary_dataset_tokenized['train'],
    eval_dataset=dictionary_dataset_tokenized['test'],
    processing_class=tokenizer,
    compute_metrics=metric,
)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight       

In [83]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,F1
1,1.317097,1.149898,0.661954
2,0.919933,0.906309,0.736148
3,0.516252,0.998221,0.761344
4,0.336420,1.011564,0.764846


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


TrainOutput(global_step=360, training_loss=0.7724253495534261, metrics={'train_runtime': 66.464, 'train_samples_per_second': 343.585, 'train_steps_per_second': 5.416, 'total_flos': 355585693376904.0, 'train_loss': 0.7724253495534261, 'epoch': 4.0})

In [84]:
preds = trainer.predict(eval_dataset)

In [85]:
preds

PredictionOutput(predictions=array([[-1.625,  1.687],
       [-1.398,  1.073],
       [-1.553,  1.746],
       ...,
       [-2.084,  2.068],
       [-1.612,  1.983],
       [-1.553,  1.524]], dtype=float16), label_ids=None, metrics={'test_runtime': 1.9552, 'test_samples_per_second': 1668.913, 'test_steps_per_second': 13.298})

In [86]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 1.0115635395050049,
 'eval_f1': 0.7648456057007126,
 'eval_runtime': 1.2313,
 'eval_samples_per_second': 1546.312,
 'eval_steps_per_second': 12.182,
 'epoch': 4.0}

In [108]:
val_ds = dictionary_dataset_tokenized['test']
preds_val = trainer.predict(val_ds)
pred_classes = np.argmax(preds_val.predictions, axis=1)
n = 5  # ile przykładów chcesz zobaczyć
for i in range(n):
    text = val_ds[i]['input']
    target = val_ds[i]['labels']
    prediction = pred_classes[i]
    print(f"TEXT: {text}\nTARGET: {target} | PREDICTION: {prediction}\n{'-'*40}")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


TEXT: Massive Typhoon heads toward Taiwan. http://t.co/Na2Ey64Vsg
TARGET: 1 | PREDICTION: 1
----------------------------------------
TEXT: Had lunch with Stewart &amp; Julian only a couple of hours earlier. Good to finally find out what happened to them. http://t.co/AnP9g6NjFd
TARGET: 0 | PREDICTION: 0
----------------------------------------
TEXT: @DoriCreates @alhanda seems gov moonbeam between tokes blames bush for all the fires.
TARGET: 1 | PREDICTION: 1
----------------------------------------
TEXT: 24 killed in two simultaneous rail crash as acute floods derail the two trains #India #mumbai... http://t.co/b0ZwI0qPTU
TARGET: 1 | PREDICTION: 1
----------------------------------------
TEXT: #3: TITAN WarriorCord 100 Feet - Authentic Military 550 Paracord - MIL-C-5040-H Type III 7 Strand 5/16' di... http://t.co/EEjRMKtJ0R
TARGET: 0 | PREDICTION: 0
----------------------------------------
